# Sales Intelligence EDA

This notebook analyzes B2B sales deal data to understand why win rates may be declining
despite healthy pipeline volume.

The goal is to surface actionable insights that sales leadership can use to
improve revenue outcomes.


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
plt.style.use("default")


In [3]:
# Load sales data
df = pd.read_csv("../data/skygini_sales_data.csv")

# Preview data
df.head()


,deal_id,created_date,closed_date,sales_rep_id,industry,region,product_type,lead_source,deal_stage,deal_amount,sales_cycle_days,outcome
0,D00001,2023-11-24,2023-12-15,rep_22,SaaS,North America,Enterprise,Referral,Qualified,4253,21,Won
1,D00002,2023-01-17,2023-01-27,rep_7,SaaS,India,Core,Referral,Closed,3905,10,Won
2,D00003,2023-10-29,2023-12-10,rep_5,HealthTech,APAC,Core,Inbound,Proposal,10615,42,Lost
3,D00004,2023-07-14,2023-08-02,rep_18,FinTech,India,Core,Partner,Negotiation,4817,19,Won
4,D00005,2024-02-29,2024-05-26,rep_2,HealthTech,APAC,Core,Outbound,Qualified,45203,87,Lost


In [4]:
# Shape and columns
df.shape, df.columns


((5000, 12),
 Index(['deal_id', 'created_date', 'closed_date', 'sales_rep_id', 'industry',
        'region', 'product_type', 'lead_source', 'deal_stage', 'deal_amount',
        'sales_cycle_days', 'outcome'],
       dtype='object'))

In [5]:
# Data types
df.dtypes


,0
deal_id,object
created_date,object
closed_date,object
sales_rep_id,object
industry,object
region,object
product_type,object
lead_source,object
deal_stage,object
deal_amount,int64


In [6]:
# Missing values per column
df.isnull().sum()


,0
deal_id,0
created_date,0
closed_date,0
sales_rep_id,0
industry,0
region,0
product_type,0
lead_source,0
deal_stage,0
deal_amount,0


In [7]:
# Win vs Loss distribution
df['outcome'].value_counts()


,count
outcome,
Lost,2737
Won,2263


### Deal Outcomes

This shows the distribution of won vs lost deals.
A higher proportion of losses may indicate pipeline quality or execution issues.


In [8]:
win_rate_by_stage = (
    df.groupby("deal_stage")["outcome"]
    .value_counts(normalize=True)
    .unstack()
)

win_rate_by_stage


outcome,Lost,Won
deal_stage,,
Closed,0.532598,0.467402
Demo,0.541707,0.458293
Negotiation,0.533668,0.466332
Proposal,0.553023,0.446977
Qualified,0.577406,0.422594


### Deal Outcomes

This shows the distribution of won vs lost deals.
A higher proportion of losses may indicate pipeline quality or execution issues.



In [9]:
df.groupby("outcome")["deal_amount"].describe()


,count,mean,std,min,25%,50%,75%,max
outcome,,,,,,,,
Lost,2737.0,25883.516989,27488.326854,2002.0,6615.0,13804.0,37082.0,99996.0
Won,2263.0,26773.874503,27928.597391,2006.0,6602.0,14623.0,40565.0,100000.0


### Deal Amount vs Outcome

This checks whether higher-value deals are more likely to be lost,
which may suggest over-optimistic pipeline creation.


In [10]:
# Win rate by region
df.groupby("region")["outcome"].value_counts(normalize=True).unstack()


outcome,Lost,Won
region,,
APAC,0.550725,0.449275
Europe,0.544201,0.455799
India,0.542768,0.457232
North America,0.552058,0.447942


In [11]:
# Win rate by industry
df.groupby("industry")["outcome"].value_counts(normalize=True).unstack()


outcome,Lost,Won
industry,,
Ecommerce,0.550943,0.449057
EdTech,0.558468,0.441532
FinTech,0.522946,0.477054
HealthTech,0.554455,0.445545
SaaS,0.548452,0.451548


In [ ]:
# Win rate by lead source
df.groupby("lead_source")["outcome"].value_counts(normalize=True).unstack()


In [12]:
# Convert dates
df["created_date"] = pd.to_datetime(df["created_date"])
df["closed_date"] = pd.to_datetime(df["closed_date"])

# Sales cycle length
df["days_to_close"] = (df["closed_date"] - df["created_date"]).dt.days

df.groupby("outcome")["days_to_close"].describe()


,count,mean,std,min,25%,50%,75%,max
outcome,,,,,,,,
Lost,2737.0,64.230179,32.545913,7.0,37.0,64.0,93.0,120.0
Won,2263.0,63.173221,32.952274,7.0,34.0,63.0,91.5,120.0


### Sales Cycle Length

Longer sales cycles often correlate with lower win probability.
This helps identify deals that should be deprioritized early.


In [13]:
# Custom Metric 1: Pipeline Efficiency
pipeline_efficiency = (
    df[df["outcome"] == "won"]["deal_amount"].sum() /
    df["deal_amount"].sum()
)

pipeline_efficiency


np.float64(0.0)

In [14]:
# Custom Metric 2: Loss Concentration
loss_concentration = (
    df[df["outcome"] == "lost"]["deal_amount"].sum() /
    df["deal_amount"].sum()
)

loss_concentration


np.float64(0.0)

### Custom Metrics

**Pipeline Efficiency** shows how much of pipeline value actually converts to revenue.

**Loss Concentration** shows how much potential revenue is lost, even with a healthy pipeline.


## Key Insights

- Win rates decline significantly in later deal stages
- High deal value does not guarantee successful closure
- Longer sales cycles are strongly associated with lost deals
- Certain regions, industries, and lead sources underperform consistently

## Business Recommendations

- Improve early-stage deal qualification
- Focus sales effort on faster-closing opportunities
- Actively clean pipeline to remove low-probability deals
- Coach sales reps on stages with highest loss rates


In [16]:
# Basic cleaned dataset
df_clean = df.dropna()

df_clean.to_csv("../data/processed/cleaned_sales_data.csv", index=False)
